# KernelPack RAG — Pipeline Inspection

A window into the live Qdrant vector DB and the current state of the pipeline.
Every number here is read from Qdrant or computed with the **real**
`retrieve.py` functions. Nothing is reimplemented and nothing that already
lives in `config.py` is hardcoded.

**How to use this notebook**

- **Run Section 0 (Setup) first.** It opens the Qdrant client, derives all
  collection/space names from `COLLECTIONS_CONFIG`, and defines small print
  helpers. Everything else depends on it.
- After Setup, **each analysis section is independent**. Run them in any order.
- Sections that need a query embedder build `JinaCodeEmbedder` lazily and cache
  it, so the (heavy) model loads at most once per kernel.

Sections:

0. Setup
1. Collection overview
2. Sample point inspector
3. Payload field coverage
4. Retrieval smoke test
5. Eval readiness check
6. Experiment readiness map

## 0. Setup

Opens the Qdrant client and pulls every required name from the codebase:
collection names from `kernelpack_rag.ingest`, declared vector/sparse spaces
from `kernelpack_rag.config.COLLECTIONS_CONFIG`. No collection or space name is
typed by hand below. (if the schema changes, this notebook follows it)

The query embedder (`JinaCodeEmbedder`) is built lazily by `query_embedder()`
because it loads a transformer model; sections that don't retrieve never pay
for it.

In [2]:
import json
import sys
from pathlib import Path

# Make the kernelpack_rag package importable regardless of where Jupyter started.
RAG_ROOT = Path('/Users/jordanchambers/public-projects/scientific-codebase-rag/codebase-rag')
if str(RAG_ROOT) not in sys.path:
    sys.path.insert(0, str(RAG_ROOT))

from qdrant_client import QdrantClient, models

from kernelpack_rag.config import COLLECTIONS_CONFIG
from kernelpack_rag.ingest import CODE_COLLECTION, PAPERS_COLLECTION
from kernelpack_rag import retrieve

# Single shared client for the whole notebook.
client = QdrantClient(host='localhost', port=6333)

# Names derived from the codebase — never hardcoded here.
CODE_DENSE_SPACES = sorted(COLLECTIONS_CONFIG[CODE_COLLECTION]['vectors'])
CODE_SPARSE_SPACES = sorted(COLLECTIONS_CONFIG[CODE_COLLECTION]['sparse_vectors'])
PAPER_DENSE_SPACES = sorted(COLLECTIONS_CONFIG[PAPERS_COLLECTION]['vectors'])
PAPER_SPARSE_SPACES = sorted(COLLECTIONS_CONFIG[PAPERS_COLLECTION]['sparse_vectors'])

print('Collections (from kernelpack_rag.ingest):')
print(f'  code   : {CODE_COLLECTION}')
print(f'  papers : {PAPERS_COLLECTION}')
print(f'Qdrant up: {client.collection_exists(CODE_COLLECTION)}')


# --- lazy query embedder ----------------------------------------------------
_QUERY_EMBEDDER = None

def query_embedder():
    '''Build and cache the JinaCode embedder used for the populated ctx__jinacode
    space. Loads the transformer model on first call only.'''
    global _QUERY_EMBEDDER
    if _QUERY_EMBEDDER is None:
        from kernelpack_rag.embed.jinacode import JinaCodeEmbedder
        print('Loading JinaCodeEmbedder (first use)...')
        _QUERY_EMBEDDER = JinaCodeEmbedder()
    return _QUERY_EMBEDDER


# --- small helpers ----------------------------------------------------------
def scroll_all(collection, scroll_filter=None, with_vectors=False, limit=None):
    '''Yield every point matching an optional filter, paging through scroll.'''
    offset = None
    seen = 0
    while True:
        page = min(256, limit - seen) if limit is not None else 256
        points, offset = client.scroll(
            collection_name=collection,
            scroll_filter=scroll_filter,
            limit=page,
            offset=offset,
            with_payload=True,
            with_vectors=with_vectors,
        )
        for p in points:
            yield p
        seen += len(points)
        if offset is None or (limit is not None and seen >= limit):
            break


def vector_is_populated(point, space):
    '''True if `space` actually carries a vector on this point (dense or sparse).'''
    vectors = getattr(point, 'vector', None)
    if not isinstance(vectors, dict):
        return False
    v = vectors.get(space)
    if v is None:
        return False
    if hasattr(v, 'indices'):       # sparse vector
        return bool(v.indices)
    return bool(v)


def granularity_filter(value):
    return models.Filter(must=[models.FieldCondition(
        key='granularity', match=models.MatchValue(value=value))])


def chunk_label(payload):
    '''Human-readable 'Module.Class.func' style label for a code chunk.'''
    parts = [payload.get('module') or '?']
    cls = payload.get('parent_class')
    fn = payload.get('function_name')
    tail = '.'.join(x for x in (cls, fn) if x)
    return f"{parts[0]}::{tail}" if tail else parts[0]


def print_candidate(rank, cand, text_chars=160):
    p = cand.payload
    text = (p.get('text') or '').strip().replace('\n', ' ')
    print(f'  #{rank}  score={cand.fused_score:.5f}  [{p.get("granularity","?"):6s}] {chunk_label(p)}')
    print(f'       file: {p.get("source_file","?")}')
    print(f'       text: {text[:text_chars]}{"..." if len(text) > text_chars else ""}')


print('\nSetup complete. Run any section below in any order.')

Collections (from kernelpack_rag.ingest):
  code   : kernelpack_code
  papers : kernelpack_papers
Qdrant up: True

Setup complete. Run any section below in any order.


## 1. Collection overview

For every collection declared in `COLLECTIONS_CONFIG`, this shows the point
count, all declared named vector spaces (dense + sparse), and (by sampling
real points and inspecting their stored vectors) **which spaces are actually
populated vs. empty**. A space can be declared in the schema yet hold no
vectors if no ingest run has filled it; that distinction is the whole point of
this section.

In [3]:
SAMPLE_FOR_POPULATION = 256  # points sampled per collection to detect populated spaces

for collection in COLLECTIONS_CONFIG:
    info = client.get_collection(collection)
    declared_dense = sorted(COLLECTIONS_CONFIG[collection]['vectors'])
    declared_sparse = sorted(COLLECTIONS_CONFIG[collection]['sparse_vectors'])
    count = info.points_count

    print('=' * 78)
    print(f'{collection}   —   {count} points')
    print('=' * 78)

    # Sample points with their vectors to see which spaces are filled.
    sample = list(scroll_all(collection, with_vectors=True, limit=SAMPLE_FOR_POPULATION))
    n = len(sample)

    if n == 0:
        print('  (no points — every declared space is empty)')
        for space in declared_dense + declared_sparse:
            kind = 'sparse' if space in declared_sparse else 'dense'
            print(f'    [empty ] {space}  ({kind})')
        print()
        continue

    print(f'  Sampled {n} points to detect populated spaces.\n')
    for space in declared_dense + declared_sparse:
        kind = 'sparse' if space in declared_sparse else 'dense'
        filled = sum(1 for p in sample if vector_is_populated(p, space))
        if filled == 0:
            status = 'EMPTY  '
        elif filled == n:
            status = 'FULL   '
        else:
            status = 'PARTIAL'
        print(f'    [{status}] {space:22s} ({kind:6s})  {filled}/{n} populated in sample')
    print()

kernelpack_code   —   1429 points
  Sampled 256 points to detect populated spaces.

    [EMPTY  ] code__jinacode         (dense )  0/256 populated in sample
    [EMPTY  ] code__qwen3            (dense )  0/256 populated in sample
    [EMPTY  ] code__unixcoder        (dense )  0/256 populated in sample
    [EMPTY  ] codecom__jinacode      (dense )  0/256 populated in sample
    [EMPTY  ] codecom__qwen3         (dense )  0/256 populated in sample
    [EMPTY  ] codecom__unixcoder     (dense )  0/256 populated in sample
    [EMPTY  ] com__jinacode          (dense )  0/256 populated in sample
    [EMPTY  ] com__qwen3             (dense )  0/256 populated in sample
    [EMPTY  ] com__unixcoder         (dense )  0/256 populated in sample
    [FULL   ] ctx__jinacode          (dense )  256/256 populated in sample
    [EMPTY  ] ctx__qwen3             (dense )  0/256 populated in sample
    [EMPTY  ] ctx__unixcoder         (dense )  0/256 populated in sample
    [EMPTY  ] math__qwen3            (

## 2. Sample point inspector

Pulls one **coarse** chunk and one **fine** chunk and shows everything about
them: the full payload (all fields), which vector spaces carry a vector, and –
called out explicitly – the `context_header` and `llm_summary` fields, since
those are the enrichment fields that feed the `ctx__*` representations.

In [4]:
def first_point(granularity):
    for p in scroll_all(CODE_COLLECTION, granularity_filter(granularity),
                        with_vectors=True, limit=1):
        return p
    return None

for granularity in ('coarse', 'fine'):
    point = first_point(granularity)
    print('#' * 78)
    print(f'# {granularity.upper()} sample point  —  id={getattr(point, "id", None)}')
    print('#' * 78)
    if point is None:
        print(f'  No {granularity} points found.\n')
        continue

    payload = point.payload or {}

    # Populated vector spaces for this specific point.
    vectors = point.vector if isinstance(point.vector, dict) else {}
    populated = [s for s in sorted(vectors) if vector_is_populated(point, s)]
    empty = [s for s in sorted(vectors) if not vector_is_populated(point, s)]
    print('\n-- vector spaces --')
    print(f'  populated: {populated}')
    print(f'  present-but-empty: {empty if empty else "(none)"}')

    # context_header and llm_summary called out explicitly.
    print('\n-- context_header --')
    print('  ' + (payload.get('context_header') or '(empty)').replace('\n', '\n  '))
    print('\n-- llm_summary --')
    print('  ' + (payload.get('llm_summary') or '(empty)').replace('\n', '\n  '))

    # Full payload, all fields.
    print('\n-- full payload --')
    for key in sorted(payload):
        value = payload[key]
        rendered = json.dumps(value, default=str)
        if len(rendered) > 300:
            rendered = rendered[:300] + ' ...'
        print(f'  {key:16s}: {rendered}')
    print()

##############################################################################
# COARSE sample point  —  id=018fd72c-2190-5329-bf30-8e37ffac6b4c
##############################################################################

-- vector spaces --
  populated: ['bm25_code', 'ctx__jinacode']
  present-but-empty: (none)

-- context_header --
  # import numpy as np
  # above: def _flatten_strip_clouds(local_clouds: list[np.ndarray], x_min: np.ndarray, x_max: np.ndarray) -> np.ndarray:
  # below: def _ensure_geometry_level_set(geometry: object, auto_build: bool) -> RBFLevelSet:

-- llm_summary --
  This function filters a set of points (nodes) by their position relative to a given geometry, using the geometry's level set representation to determine which points lie inside or outside. Key parameters include `keep`, which specifies whether to retain points inside or outside the geometry, and tolerance and clearance values that control how close points can be to the boundary. It returns the filt

## 3. Payload field coverage

Across **all coarse points**, what fraction carry `math_terms`, `cross_refs`,
and `llm_summary`? These drive downstream experiments: `math_terms` powers the
payload-filtered retrieval (E2.4a), `cross_refs`/`cross_ref_ids` power
cross-reference expansion (E2.4c), and `llm_summary` feeds the summary
representation. A few real example values are shown so you can sanity-check
extraction quality.

In [5]:
coarse = list(scroll_all(CODE_COLLECTION, granularity_filter('coarse')))
n = len(coarse)
print(f'Coarse points: {n}\n')

def coverage(field):
    have = [p for p in coarse if p.payload.get(field)]
    pct = (100.0 * len(have) / n) if n else 0.0
    return have, pct

for field in ('math_terms', 'cross_refs', 'cross_ref_ids', 'llm_summary'):
    have, pct = coverage(field)
    print(f'  {field:16s}: {len(have):4d}/{n}  ({pct:5.1f}%)')

print('\n-- example math_terms values --')
for p in [p for p in coarse if p.payload.get('math_terms')][:5]:
    print(f'  {chunk_label(p.payload)}')
    print(f'      {p.payload["math_terms"]}')

print('\n-- example cross_refs values --')
for p in [p for p in coarse if p.payload.get('cross_refs')][:3]:
    refs = p.payload['cross_refs']
    print(f'  {chunk_label(p.payload)}  ({len(refs)} refs)')
    for r in refs[:4]:
        print(f'      {r}')
    if len(refs) > 4:
        print(f'      ... (+{len(refs) - 4} more)')

Coarse points: 399

  math_terms      :  196/399  ( 49.1%)
  cross_refs      :  278/399  ( 69.7%)
  cross_ref_ids   :  278/399  ( 69.7%)
  llm_summary     :  399/399  (100.0%)

-- example math_terms values --
  kernelpack.nodes.core::clip_points_by_geometry
      ['level set']
  kernelpack.solvers.detail.incompressible_euler_bdf_backend::IncompressibleEulerBDFBackend._solve_bdf_step
      ['divergence', 'gmres']
  kernelpack.solvers.pu_sl_advection::query_patch_ids_for_advection
      ['advection']
  kernelpack.solvers._pu::pu_patch_geometry
      ['stencil']
  kernelpack.rbffd.core::FDODiffOp.assemble_op
      ['stencil']

-- example cross_refs values --
  kernelpack.nodes.core::clip_points_by_geometry  (6 refs)
      kernelpack.divfree.core.DivFreePHSInterpolant.evaluate
      kernelpack.divfree.core.LocalDivFreeInterpolator.evaluate
      kernelpack.geometry.core.RBFLevelSet.evaluate
      kernelpack.nodes.core._ensure_geometry_level_set
      ... (+2 more)
  kernelpack.solvers.deta

## 4. Retrieval smoke test

Runs the real `retrieve.hybrid()` (dense `ctx__jinacode` + sparse `bm25_code`,
fused with RRF) on three representative queries: one API-tier, one
workflow-tier, one math-heavy. For each, the top 3 results show the chunk text,
fused score, and which module/function the chunk came from.

Signature actually being called:

```python
retrieve.hybrid(query, *, client, collection, embedder,
                k=10, space="ctx__jinacode", reranker=None, injected_ids=None)
```

In [6]:
SMOKE_QUERIES = [
    ('api',        'How do I solve a stationary Poisson problem with mixed boundary conditions?'),
    ('workflow',   'Build RBF-FD stencils and assemble the differentiation operator for a node set.'),
    ('math-heavy', 'Compute the polyharmonic spline kernel and its Laplacian for divergence-free interpolation.'),
]

embedder = query_embedder()

for tier, q in SMOKE_QUERIES:
    print('=' * 78)
    print(f'[{tier}] {q}')
    print('=' * 78)
    results = retrieve.hybrid(
        q,
        client=client,
        collection=CODE_COLLECTION,
        embedder=embedder,
        k=10,
    )
    if not results:
        print('  (no results)\n')
        continue
    for rank, cand in enumerate(results[:3], start=1):
        print_candidate(rank, cand)
    print()

/Users/jordanchambers/public-projects/scientific-codebase-rag/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading JinaCodeEmbedder (first use)...


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 7226.70it/s]


[api] How do I solve a stationary Poisson problem with mixed boundary conditions?
  #1  score=0.50000  [coarse] kernelpack.nodes.core::_parse_poisson_options
       file: /Users/jordanchambers/public-projects/kernelpack-python/src/kernelpack/nodes/core.py
       text: def _parse_poisson_options(     radius_or_func: float | Callable[[np.ndarray, float], float],     x_min: np.ndarray,     x_max: np.ndarray,     *,     attempts:...
  #2  score=0.50000  [fine  ] kernelpack.solvers.poisson::PoissonSolver.solve
       file: /Users/jordanchambers/public-projects/kernelpack-python/src/kernelpack/solvers/poisson.py
       text: self.bc = bc_assembler.get_op().tocsr()         rhs_target = evaluate_node_callback(forcing, self.x, "forcing")         rhs_boundary = evaluate_boundary_values(...
  #3  score=0.33333  [coarse] kernelpack.nodes.core::_has_conflicting_neighbor
       file: /Users/jordanchambers/public-projects/kernelpack-python/src/kernelpack/nodes/core.py
       text: def _has_conflictin

## 5. Eval readiness check

Loads the labeled `solvers_qa.json`, runs every query through
`retrieve.hybrid()` (k=10), and reports **recall@3 / @5 / @10**. A result counts
as a hit when a retrieved chunk comes from the labeled `source_file` **and** its
symbol matches the labeled `source_symbol` (coarse or fine chunk of that
function both count). For every miss, the top 5 chunks that *did* rank are shown
so you can see what the retriever preferred instead.

In [7]:
# Locate the QA file relative to the repo without hardcoding an absolute path.
QA_CANDIDATES = [
    RAG_ROOT.parent / 'experiments' / 'qa_pairs' / 'solvers_qa.json',
    RAG_ROOT / 'qa_pairs' / 'solvers_qa.json',
]
QA_PATH = next((p for p in QA_CANDIDATES if p.exists()), None)
assert QA_PATH is not None, f'solvers_qa.json not found in: {QA_CANDIDATES}'
qa_pairs = json.loads(QA_PATH.read_text())
print(f'Loaded {len(qa_pairs)} QA pairs from {QA_PATH}\n')


def is_relevant(payload, qa):
    '''A retrieved chunk is relevant if it lives in the labeled source file and
    its symbol matches the labeled source_symbol.'''
    src = payload.get('source_file') or ''
    if not src.endswith(qa['source_file']):
        return False
    cls = payload.get('parent_class')
    fn = payload.get('function_name') or ''
    symbol = '.'.join(x for x in (cls, fn) if x)
    target = qa['source_symbol']
    return symbol == target or fn == target.split('.')[-1]


def first_hit_rank(results, qa):
    for rank, cand in enumerate(results, start=1):
        if is_relevant(cand.payload, qa):
            return rank
    return None


embedder = query_embedder()
K_LEVELS = (3, 5, 10)
hits = {k: 0 for k in K_LEVELS}
misses = []

for qa in qa_pairs:
    results = retrieve.hybrid(
        qa['query'],
        client=client,
        collection=CODE_COLLECTION,
        embedder=embedder,
        k=max(K_LEVELS),
    )
    rank = first_hit_rank(results, qa)
    for k in K_LEVELS:
        if rank is not None and rank <= k:
            hits[k] += 1
    if rank is None or rank > max(K_LEVELS):
        misses.append((qa, results))

n = len(qa_pairs)
print('Recall (target = labeled source_symbol in labeled source_file):')
for k in K_LEVELS:
    print(f'  recall@{k:<2d}: {hits[k]}/{n}  ({100.0 * hits[k] / n:5.1f}%)')

print(f'\nMisses (not retrieved in top {max(K_LEVELS)}): {len(misses)}')
for qa, results in misses:
    print('-' * 78)
    print(f'[{qa["tier"]}] {qa["query"]}')
    print(f'  expected: {qa["source_symbol"]}  ({qa["source_file"]})')
    print('  top 5 that DID rank:')
    for rank, cand in enumerate(results[:5], start=1):
        print_candidate(rank, cand, text_chars=90)

Loaded 10 QA pairs from /Users/jordanchambers/public-projects/scientific-codebase-rag/experiments/qa_pairs/solvers_qa.json

Recall (target = labeled source_symbol in labeled source_file):
  recall@3 : 7/10  ( 70.0%)
  recall@5 : 7/10  ( 70.0%)
  recall@10: 9/10  ( 90.0%)

Misses (not retrieved in top 10): 1
------------------------------------------------------------------------------
[workflow] For a concentration matrix with one column per species, how are independent diffusion channels created before time stepping?
  expected: MultiSpeciesDiffusionSolver.set_initial_state  (src/kernelpack/solvers/multispecies_diffusion.py)
  top 5 that DID rank:
  #1  score=0.64286  [coarse] kernelpack.solvers.heterogeneous_multispecies_diffusion::HeterogeneousMultiSpeciesPUDiffusionSolver._step_columns
       file: /Users/jordanchambers/public-projects/kernelpack-python/src/kernelpack/solvers/heterogeneous_multispecies_diffusion.py
       text: def _step_columns(         self,         t: float,    

## 6. Experiment readiness map

One subsection per planned experiment. Each states: what it tests, what is
already built and ready, the exact function call that kicks it off (real
signature from `retrieve.py`), and what is blocked and why.

The probe cell below reports the live ground truth the rest of this section
relies on (which code spaces actually hold vectors and whether the papers
collection has points) so the readiness claims stay honest if you re-run after
a new ingest.

In [8]:
# --- live readiness probe ---------------------------------------------------
sample = list(scroll_all(CODE_COLLECTION, with_vectors=True, limit=256))
declared = CODE_DENSE_SPACES + CODE_SPARSE_SPACES
space_state = {}
for space in declared:
    filled = sum(1 for p in sample if vector_is_populated(p, space))
    space_state[space] = 'POPULATED' if filled else 'EMPTY'

papers_count = client.get_collection(PAPERS_COLLECTION).points_count

print('Code vector spaces (sample of {} points):'.format(len(sample)))
for space in declared:
    print(f'  {space_state[space]:10s}  {space}')
print(f'\n{PAPERS_COLLECTION} points: {papers_count}')

# Quick helpers the markdown narrative refers to:
ready_spaces = sorted(s for s, st in space_state.items() if st == 'POPULATED')
empty_spaces = sorted(s for s, st in space_state.items() if st == 'EMPTY')
print(f'\nReady spaces : {ready_spaces}')
print(f'Empty spaces : {empty_spaces}')
print(f'Papers ready : {papers_count > 0}')

Code vector spaces (sample of 256 points):
  EMPTY       code__jinacode
  EMPTY       code__qwen3
  EMPTY       code__unixcoder
  EMPTY       codecom__jinacode
  EMPTY       codecom__qwen3
  EMPTY       codecom__unixcoder
  EMPTY       com__jinacode
  EMPTY       com__qwen3
  EMPTY       com__unixcoder
  POPULATED   ctx__jinacode
  EMPTY       ctx__qwen3
  EMPTY       ctx__unixcoder
  EMPTY       math__qwen3
  EMPTY       summary__qwen3
  POPULATED   bm25_code

kernelpack_papers points: 0

Ready spaces : ['bm25_code', 'ctx__jinacode']
Empty spaces : ['code__jinacode', 'code__qwen3', 'code__unixcoder', 'codecom__jinacode', 'codecom__qwen3', 'codecom__unixcoder', 'com__jinacode', 'com__qwen3', 'com__unixcoder', 'ctx__qwen3', 'ctx__unixcoder', 'math__qwen3', 'summary__qwen3']
Papers ready : False


### E1.2 — Multi-model ablation (`ctx__qwen3`, `ctx__unixcoder`)

- **Tests:** does swapping the embedding model behind the *same* context
  representation (`ctx`) change retrieval quality? Compare `ctx__jinacode`
  (current default) vs. `ctx__qwen3` vs. `ctx__unixcoder`.
- **Already built:** `retrieve.hybrid()` takes a `space=` argument and an
  `embedder=`, so no retrieval code changes are needed — just point it at a
  different dense space with the matching embedder.

  ```python
  from kernelpack_rag.embed.qwen import QwenEmbedder
  retrieve.hybrid(query, client=client, collection=CODE_COLLECTION,
                  embedder=QwenEmbedder(), k=10, space="ctx__qwen3")
  ```
- **Blocked:** `ctx__qwen3` and `ctx__unixcoder` are **declared but empty** (see
  probe above) — only `ctx__jinacode` was populated by the ingest run. Populate
  them first:

  ```bash
  python -m kernelpack_rag.ingest --source /abs/path/to/kernelpack \
      --spaces ctx__qwen3 ctx__unixcoder
  ```

### E2.3 — Representation ablation (`code__*`, `com__*`, `codecom__*`)

- **Tests:** which *representation* of a chunk embeds best — raw code (`code`),
  comments/docstring only (`com`), or code+comments (`codecom`) — holding the
  model fixed. Compare against the `ctx` baseline.
- **Already built:** the representations are constructed at ingest time by
  `kernelpack_rag.embed.representations.build_representation`, and
  `retrieve.hybrid(..., space=...)` can query any of these spaces directly.
- **Blocked:** every `code__*`, `com__*`, and `codecom__*` space is **declared
  but empty** (probe above). Also note the ingest pipeline only writes the
  non-`ctx` representations for **coarse** points (`_code_vectors` restricts
  fine points to `ctx__jinacode` + `bm25_code`), so this ablation is coarse-only
  until that changes. Populate the coarse representations with:

  ```bash
  python -m kernelpack_rag.ingest --source /abs/path/to/kernelpack \
      --spaces code__jinacode com__jinacode codecom__jinacode
  ```

### E2.4a — `math_terms` payload filtering (`retrieve.hybrid_filtered()`)

- **Tests:** does constraining hybrid retrieval to chunks tagged with specific
  `math_terms` improve precision on math-heavy queries?
- **Already built and READY:** runs on the populated `ctx__jinacode` +
  `bm25_code` spaces, and the `math_terms` payload field is present on ~half of
  coarse points (see Section 3), which is what the filter keys on.

  Real signature / call:

  ```python
  retrieve.hybrid_filtered(
      query,
      math_terms=["gmres", "divergence"],
      client=client, collection=CODE_COLLECTION,
      embedder=query_embedder(), k=10,
      space="ctx__jinacode", reranker=None,
  )
  ```
  Returns `[]` if `math_terms` is empty (by design).

In [9]:
# E2.4a live demo — payload-filtered hybrid on a math_terms tag.
demo = retrieve.hybrid_filtered(
    'How is the GMRES solve set up for the pressure system?',
    math_terms=['gmres', 'divergence'],
    client=client,
    collection=CODE_COLLECTION,
    embedder=query_embedder(),
    k=5,
)
print(f'hybrid_filtered returned {len(demo)} candidates (filtered to math_terms)')
for rank, cand in enumerate(demo[:3], start=1):
    print_candidate(rank, cand, text_chars=90)

hybrid_filtered returned 5 candidates (filtered to math_terms)
  #1  score=0.55882  [coarse] kernelpack.solvers.poisson::PoissonSolver.solve
       file: /Users/jordanchambers/public-projects/kernelpack-python/src/kernelpack/solvers/poisson.py
       text: def solve(         self,         forcing: Callable[..., np.ndarray] | np.ndarray,         ...
  #2  score=0.50000  [fine  ] kernelpack.solvers.nonlinear_variable_poisson::_gmres_step
       file: /Users/jordanchambers/public-projects/kernelpack-python/src/kernelpack/solvers/nonlinear_variable_poisson.py
       text: return gmres_with_preconditioner(         jacobian,         rhs,         None,         ilu...
  #3  score=0.33333  [coarse] kernelpack.solvers.detail.incompressible_euler_bdf_backend::IncompressibleEulerBDFBackend._solve_bdf_step
       file: /Users/jordanchambers/public-projects/kernelpack-python/src/kernelpack/solvers/detail/incompressible_euler_bdf_backend.py
       text: def _solve_bdf_step(self, alpha0: float, histor

### E2.4c — Cross-reference expansion (`retrieve.expand_cross_refs()`)

- **Tests:** does pulling in chunks that a retrieved chunk *cross-references*
  (via `cross_ref_ids`) add useful context that pure similarity search misses?
- **Already built and READY:** `cross_ref_ids` is populated on 100% of coarse
  points (Section 3). The function operates on an existing candidate list and
  fetches referenced points by id — no embedder or extra space needed.

  Real signature:

  ```python
  base = retrieve.hybrid(query, client=client, collection=CODE_COLLECTION,
                         embedder=query_embedder(), k=10)
  retrieve.expand_cross_refs(base, client=client,
                            collection=CODE_COLLECTION, hops=1)
  ```

In [10]:
# E2.4c live demo — expand the hybrid result set by one hop of cross-refs.
base = retrieve.hybrid(
    'assemble the boundary operator for a Poisson solve',
    client=client, collection=CODE_COLLECTION,
    embedder=query_embedder(), k=5,
)
expanded = retrieve.expand_cross_refs(
    base, client=client, collection=CODE_COLLECTION, hops=1,
)
print(f'base candidates: {len(base)}  ->  after 1-hop expansion: {len(expanded)}')
added = [c for c in expanded if c.leg_scores.get('provenance') == 'cross_ref']
print(f'added via cross_ref: {len(added)}')
for cand in added[:3]:
    print(f'  + {chunk_label(cand.payload)}')

base candidates: 5  ->  after 1-hop expansion: 26
added via cross_ref: 21
  + kernelpack.nodes.core::_default_strip_count
  + kernelpack.rbffd.core::CrossNodeDiffOp.assemble_op
  + kernelpack.rbffd.core::CrossNodeDiffOp.get_op


### E2.2 — Retrieve-small-read-big (`retrieve.fine_to_coarse()`)

- **Tests:** does searching over **fine** (statement-window) chunks for precise
  matching, then returning their **coarse** parents for full context, beat
  searching coarse chunks directly?
- **Already built and READY:** both granularities are populated on
  `ctx__jinacode` + `bm25_code` (Sections 1–2 show fine + coarse points), and
  fine chunks carry `parent_id`, which is exactly what this plan walks up.

  Real signature:

  ```python
  retrieve.fine_to_coarse(
      query, client=client, collection=CODE_COLLECTION,
      embedder=query_embedder(), k=10, reranker=None,
  )
  ```

In [11]:
# E2.2 live demo — retrieve over fine chunks, return coarse parents.
f2c = retrieve.fine_to_coarse(
    'time-stepping with a BDF scheme for the diffusion solver',
    client=client, collection=CODE_COLLECTION,
    embedder=query_embedder(), k=5,
)
print(f'fine_to_coarse returned {len(f2c)} coarse parents')
for rank, cand in enumerate(f2c[:3], start=1):
    print_candidate(rank, cand, text_chars=90)
    print(f'       (matched via child fine chunk id={cand.leg_scores.get("child_id")})')

fine_to_coarse returned 5 coarse parents
  #1  score=0.50000  [coarse] kernelpack.solvers.pu_sl_fd_advection_diffusion::PUSLFDAdvectionDiffusionSolver.bdf1_step
       file: /Users/jordanchambers/public-projects/kernelpack-python/src/kernelpack/solvers/pu_sl_fd_advection_diffusion.py
       text: def bdf1_step(         self,         t_next: float,         velocity: Callable,         rk...
       (matched via child fine chunk id=20c2ba7b-083a-5a06-934c-9b2f38d18900)
  #2  score=0.50000  [coarse] kernelpack.solvers.heterogeneous_multispecies_diffusion::HeterogeneousMultiSpeciesDiffusionSolver._step_columns
       file: /Users/jordanchambers/public-projects/kernelpack-python/src/kernelpack/solvers/heterogeneous_multispecies_diffusion.py
       text: def _step_columns(         self,         t: float,         forcing: Callable[..., np.ndarr...
       (matched via child fine chunk id=cfbf1bc2-225e-5386-8588-675b02ffdbe1)
  #3  score=0.33333  [coarse] kernelpack.solvers.heterogeneous_multispe

### E2.4b — Trimodal retrieval (`retrieve.trimodal()`)

- **Tests:** fuse three legs — context dense (`ctx__jinacode`), a math/paper
  dense leg (`math__qwen3`), and sparse (`bm25_code`) — weighted, to see if the
  math leg helps on equation-grounded queries.
- **Already built:** the function exists and fuses the three legs with weighted
  RRF.

  Real signature:

  ```python
  from kernelpack_rag.embed.jinacode import JinaCodeEmbedder
  from kernelpack_rag.embed.qwen import QwenEmbedder
  retrieve.trimodal(
      query, client=client, collection=CODE_COLLECTION,
      jinacode_embedder=JinaCodeEmbedder(), qwen_embedder=QwenEmbedder(),
      weights={"ctx": 1.0, "math": 1.0, "sparse": 1.0}, k=10,
  )
  ```
- **Blocked:** the `math__qwen3` space is **declared but empty** (probe above).
  It is only filled by `ingest._populate_math_vectors`, which copies a paper's
  `paper__qwen3` vector onto code chunks sharing its `math_terms` — and that
  runs only when papers are ingested. So this is blocked behind the **papers
  collection being empty** (see E2.5) plus a `--spaces math__qwen3` ingest run.

### E2.5 — Two-leg retrieval (`retrieve.two_leg()`)

- **Tests:** route the query first through the **papers** corpus, harvest the
  `math_terms` of the top papers, then run a `math_terms`-filtered code search —
  using the literature as a bridge into the implementation.
- **Already built:** `retrieve.two_leg()` implements exactly this hop
  (papers hybrid → collect `math_terms` → `hybrid_filtered` on code).

  Real signature:

  ```python
  retrieve.two_leg(
      query, client=client,
      code_collection=CODE_COLLECTION, papers_collection=PAPERS_COLLECTION,
      embedder=query_embedder(), k=10,
  )
  ```
- **Blocked:** `kernelpack_papers` has **0 points** (probe above), so the first
  leg returns nothing and the function returns `[]`. Ingest papers first:

  ```bash
  python -m kernelpack_rag.ingest --source /abs/path/to/kernelpack \
      --papers /abs/path/to/papers \
      --spaces paper__qwen3 bm25_paper math__qwen3
  ```